In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# load csv file
df = pd.read_csv('nitrate.csv',header=0)

<bound method NDFrame.head of                  Date   NO3N  Student_Flag
0      8/2/2016 12:29  0.071             0
1      8/2/2016 12:59  0.030             0
2      8/2/2016 13:29  0.030             0
3      8/2/2016 13:59  0.030             0
4      8/2/2016 14:29  0.030             0
...               ...    ...           ...
30785   5/7/2018 6:59  0.219             0
30786   5/7/2018 7:29  0.117             0
30787   5/7/2018 7:59  0.099             0
30788   5/7/2018 8:29  0.217             0
30789   5/7/2018 8:59  0.074             0

[30790 rows x 3 columns]>


In [ ]:
plt.figure(figsize=(15, 5))

plt.plot(
    df["Date"],
    df["NO3N"],
    label="NO3N",
    linewidth=1
)

# Overlay known anomalies
anomalies = df["Student_Flag"] == 1

plt.scatter(
    df.loc[anomalies, "Date"],
    df.loc[anomalies, "NO3N"],
    color="red",
    label="Student_Flag = 1",
    zorder=3
)

plt.xlabel("Date")
plt.ylabel("NO3N")
plt.title("NO3N Time Series")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
W = 2000  # Window size
q = 95    # Percentile threshold
values = df["NO3N"].values
n_points = len(values)

# Array to store predicted flags (0 for normal, 1 for anomaly)
pred_flags = np.zeros(n_points, dtype=int)


In [ ]:
T1 = np.percentile(values[0:W], q, method="linear")
pred_flags[0:W] = (values[0:W] >= T1).astype(int)

In [ ]:
for i in range(1, n_points - W + 1):
    window_data = values[i : i + W]
    Ti = np.percentile(window_data, q, method="linear")
    
    # Label only the new incoming point at the end of the window
    new_point_idx = i + W - 1
    if values[new_point_idx] >= Ti:
        pred_flags[new_point_idx] = 1

In [ ]:
def extract_events(flags):
    # Find start and end indices of contiguous 1s
    diff = np.diff(np.pad(flags, (1, 1), 'constant'))
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0]
    return list(zip(starts, ends))

gt_events = extract_events(df["Student_Flag"].values)  # Total = 77
pred_events = extract_events(pred_flags)

# Calculate True Positives (TP) and False Negatives (FN)
TP = 0
for start, end in gt_events:
    # An anomaly event is correctly detected if your model flags AT LEAST 1 point inside the event interval
    if np.any(pred_flags[start:end] == 1):
        TP += 1

FN = len(gt_events) - TP  # Total GT events (77) - TP
anomaly_accuracy = TP / len(gt_events)

print(f"TP: {TP} / 77")
print(f"Anomaly Accuracy: {anomaly_accuracy * 100:.2f}%")

TP: 95 / 77
Anomaly Accuracy: 70.90%
